# MUSTER 비확장 RHELM·Clone 두 설정 원 기준 수집

**별도 RTX PRO6000 GPU 런타임에서 Run all 하십시오.** 같은 VM에 FAST·Suite 또는 다른 GPU 작업이 있으면 이를 중단하지 않고 실행을 거부합니다. 기존 Drive 패키지 60개 파일을 읽으므로 새 비공개 데이터 업로드가 필요 없습니다.

대상은 비확장 RHELM 1,305문항과 Clone English 218문항뿐입니다. 원 1.0.0 SDK·모델·FP32·batch32/16·max2048과 원 생산자 검색을 그대로 쓰고 GraphHook은 설치하지 않습니다. 최대 두 자식으로 아직 없는 원 기준을 한 번씩 수집한 뒤, R이 CPU에서 상한200/400 등을 재현하고 필요한 점수만 보충합니다. LME-v2·확장 RHELM·AMA·PersonaMem을 새로 실행하지 않습니다.

완료 기준이 있으면 원 source·QID·hash를 CPU로 확인해 다시 검색하지 않습니다. 부분 결과는 새 namespace의 질문별 checkpoint를 검증해 이어갑니다. 자식 stdout을 즉시 전달하고 첫10초 뒤부터30초마다 실제 완료 질문 수·UTC·최근 속도를 표시합니다. 전체 ETA를 아직 측정하지 못했으면 `-`로 표시합니다.

새 결과는 `MyDrive/MUSTER_native_reference_two_20261116/results/`에20초 간격으로 보존합니다. 코드·메타데이터만 내장되어 있고 원문·벡터·휠·가중치는 없습니다. 답 reader/judge API 호출은0이며, 이 노트북의 검색·조립 기록은 답 정확도 측정과 별개입니다. 실제 Colab GPU 실행은 사용자 Run all 이후에 검증됩니다.


In [ ]:
from pathlib import Path
Path('/content/MUSTER_NATIVE_REFERENCE_TWO_ORIGINAL.py').write_text('"""Code-only entry: collect only the two missing original native references."""\nimport base64\nimport fcntl\nimport hashlib\nimport importlib.util\nimport io\nimport json\nimport os\nfrom pathlib import Path\nimport socket\nimport subprocess\nimport sys\nimport time\nimport zipfile\n\nCODE_PAYLOAD = "..."\nCODE_SHA = "3c51864aab5bc972be5624c2685e52db782c6226848e8b8c3fa5e732ec8c5c9c"\n\n\ndef module(path, name):\n    spec = importlib.util.spec_from_file_location(name, path)\n    result = importlib.util.module_from_spec(spec)\n    sys.modules[name] = result\n    spec.loader.exec_module(result)\n    return result\n\n\ndef unpack():\n    raw = base64.b64decode(CODE_PAYLOAD, validate=True)\n    if hashlib.sha256(raw).hexdigest() != CODE_SHA:\n        raise RuntimeError("embedded reference code ZIP differs")\n    root = Path("/content/muster_native_reference_two_code") / CODE_SHA[:16]\n    with zipfile.ZipFile(io.BytesIO(raw)) as z:\n        info = json.loads(z.read("CODE_MANIFEST.json"))\n        if set(z.namelist()) != set(info["files"]) | {"CODE_MANIFEST.json"}:\n            raise RuntimeError("reference code inventory differs")\n        for name, digest in info["files"].items():\n            if Path(name).is_absolute() or ".." in Path(name).parts:\n                raise RuntimeError("unsafe reference code path")\n            content = z.read(name)\n            if hashlib.sha256(content).hexdigest() != digest:\n                raise RuntimeError("embedded reference module differs: " + name)\n            target = root / name\n            target.parent.mkdir(parents=True, exist_ok=True)\n            target.write_bytes(content)\n    return root, info\n\n\ndef idle_runtime_required(mydrive, own_output):\n    """Read-only admission: never stop another workload or mutate its output."""\n    conflicts = []\n    markers = {"muster_static_source_search_gpu.py", "static_phase_orchestrator.py", "native_fast.py",\n               "related.py", "reference.py", "prepare_pinned_model_cache.py", "prepare_suite_assets.py"}\n    for p in Path("/proc").glob("[0-9]*/cmdline"):\n        if int(p.parent.name) == os.getpid():\n            continue\n        try:\n            argv = [v.decode(errors="replace") for v in p.read_bytes().split(b"\\0") if v]\n        except (OSError, ValueError):\n            continue\n        hit = next((Path(v).name for v in argv if Path(v).name in markers or\n                    Path(v).name.startswith(("MUSTER_GRAPH_", "MUSTER_NATIVE_")) or\n                    ("/suite/" in v and Path(v).name in ("worker.py", "orchestrator.py"))), None)\n        if hit:\n            conflicts.append({"pid": int(p.parent.name), "process": hit})\n    query = subprocess.run(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader,nounits"],\n                           check=True, capture_output=True, text=True, timeout=10)\n    gpu_pids = [int(v.strip()) for v in query.stdout.splitlines() if v.strip().isdigit()]\n    # Also detect a same-VM FAST/Suite in between its child processes.\n    for group in ("MUSTER_graph_budget_fast_20261116", "MUSTER_graph_budget_suite_20261116",\n                  "MUSTER_native_reference_two_20261116"):\n        for p in (mydrive / group / "results").glob("*/ACTIVE_OWNER.json"):\n            try:\n                state = json.loads(p.read_text())\n                running = state.get("status") == "running"\n                same_vm = state.get("host") == socket.gethostname()\n                if running and (same_vm or p.parent == own_output):\n                    pid = state.get("pid")\n                    if same_vm and type(pid) is int:\n                        try:\n                            os.kill(pid, 0)\n                        except ProcessLookupError:\n                            continue\n                    elif time.time() - state.get("heartbeat", 0) >= 180:\n                        continue\n                    conflicts.append({"pid": pid, "owner": group})\n            except FileNotFoundError:\n                continue\n    if conflicts or gpu_pids:\n        raise RuntimeError("별도 GPU 런타임이 필요합니다. 기존 작업은 중단하지 않았습니다. " +\n                           json.dumps({"active_processes": conflicts, "GPU_pids": gpu_pids}, ensure_ascii=False))\n    print("동일 VM의 FAST·Suite·다른 GPU 작업 없음: 준비를 시작합니다.", flush=True)\n\n\ndef run():\n    if sys.platform != "linux" or not Path("/content").is_dir():\n        raise RuntimeError("실제 실행은 별도 Colab GPU 런타임에서만 가능합니다")\n    code, info = unpack()\n    # Importing the frozen original file defines functions; its __main__/run is not executed.\n    helper = module(code / "metadata/fast_prepare.py", "_native_reference_original_prepare")\n    if not helper.drive_mounted():\n        raise RuntimeError("Drive를 /content/drive에 먼저 연결해야 합니다")\n    output = helper.DRIVE_ROOT / "MUSTER_native_reference_two_20261116/results" / CODE_SHA[:16]\n    with Path("/content/MUSTER_NATIVE_REFERENCE_TWO_PREPARE.lock").open("a+") as lock:\n        try:\n            fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except BlockingIOError as exc:\n            raise RuntimeError("이 VM의 두 설정 기준 수집이 이미 실행 중입니다") from exc\n        idle_runtime_required(helper.DRIVE_ROOT, output)\n        package = helper.prepare_package()\n        campaign = module(code / "campaign.py", "_native_reference_campaign")\n        reference = module(code / "reference.py", "_native_reference_preflight")\n        local = Path("/content/muster_native_reference_two_work") / CODE_SHA[:16]\n        phase = "reference_two"\n        phase_folder = local / phase\n        campaign.CheckpointMirror(phase_folder, output / phase).restore()\n        baseline = helper.DRIVE_ROOT / "MUSTER_native5_full3_static_20261115_v2/outputs/native-static-v1-718c6081b56e"\n        cache = Path("/content/muster_assembly_parallel_assets/hf")\n        check = {"package_dir": str(helper.PACKAGE_ROOT), "output_dir": str(phase_folder),\n                 "models_cache": str(cache), "baseline_dir": str(baseline)}\n        available = {}\n        for cohort in reference.COHORTS:\n            producer, args, rows, expected, contract = reference.prepare(check, cohort)\n            available[cohort] = any(reference.completed_readback(producer, source, rows, expected, contract) is not None\n                                    for source in (Path(args.out), baseline / cohort))\n        output.mkdir(parents=True, exist_ok=True)\n        print("원 기준 CPU 완료 대조:", json.dumps(available), flush=True)\n        if not all(available.values()):\n            helper.prepare_models(package, cache, output)\n        else:\n            print("두 원 기준이 모두 완성되어 모델 준비·검색을 생략합니다.", flush=True)\n        if not helper.drive_mounted():\n            raise RuntimeError("Drive 연결이 해제되어 새 작업을 시작하지 않았습니다")\n        idle_runtime_required(helper.DRIVE_ROOT, output)\n        config = {"schema": campaign.SCHEMA, "asset_dir": str(code), "output_dir": str(output),\n            "local_dir": str(local), "models_cache": str(cache), "phases": [{"id": phase, "module": "reference.py",\n                "module_sha256": info["files"]["reference.py"], "config": {\n                    "package_dir": str(helper.PACKAGE_ROOT), "output_dir": "{output}", "models_cache": "{cache}",\n                    "baseline_dir": str(baseline)}}]}\n        print("비확장 RHELM1305 + Clone218의 없는 원 기준만 최대2프로세스로 수집합니다.", flush=True)\n        print("다른 7설정·GraphHook·답 reader/judge 호출: 0. 첫10초 후 30초마다 실제 진행을 출력합니다.", flush=True)\n        print("Drive 결과:", output, flush=True)\n        result = campaign.run_campaign(config)\n        print("두 설정 원 기준 상태:", result["status"], flush=True)\n        print("다음은 R의 CPU 상한200/400 재현과 필요한 점수만 보충하는 단계입니다. 답 정확도는 미측정입니다.", flush=True)\n        return result\n\n\nif __name__ == "__main__":\n    raise SystemExit(0 if run()["status"] == "DONE" else 2)\n', encoding='utf-8')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import importlib.util
spec = importlib.util.spec_from_file_location('_native_reference_two_entry', '/content/MUSTER_NATIVE_REFERENCE_TWO_ORIGINAL.py')
entry = importlib.util.module_from_spec(spec)
spec.loader.exec_module(entry)
result = entry.run()
